# Addendum revisi, Bagian E saja (pembanding dari versi paket yang sama)
Notebook ini hanya menjalankan **Bagian E**. Bagian A–D sudah selesai dan hasilnya tersimpan di `results/revisi_C_k_mentah.csv` dan `results/revisi_D_v2b80_mentah.csv`; keduanya dibaca ulang di sini, jadi tidak perlu dijalankan lagi.

Simpan notebook ini di folder `hepatitis-leakage-reanalysis`, lalu Restart Kernel dan Run All (±25 menit).

## Bagian 0. Fungsi dan konfigurasi (disalin dari notebook utama, tanpa perubahan logika)

In [1]:
# ===== SATU-SATUNYA PENGATURAN YANG MUNGKIN PERLU DIUBAH =====
# Isi dengan lokasi folder 'hepatitis-leakage-reanalysis' di komputer masing-masing.
# Jika notebook dibuka langsung dari folder itu, pengaturan ini tidak dipakai (folder terdeteksi otomatis).
FOLDER_REPO = r"F:\ARTIKEL RAMEAN 2 -HEPATIITIS\hepatitis-leakage-reanalysis"
NAMA_PELAKSANA = "Ade"      # tulis nama yang menjalankan, misalnya "Ade" atau "Ziad"; dicatat di revisi_konfigurasi.json

import os
def _folder_sah(f):
    return all(os.path.exists(os.path.join(f, *b)) for b in
               [("data", "hcv", "hcvdat0.csv"), ("data", "hepatitis", "hepatitis.data"), ("results", "hasil_semua_mentah.csv"), ("results", "revisi_C_k_mentah.csv"), ("results", "revisi_D_v2b80_mentah.csv")])
for _kandidat in (os.getcwd(), FOLDER_REPO):
    if _folder_sah(_kandidat):
        os.chdir(_kandidat)
        break
else:
    raise FileNotFoundError("Folder repositori tidak ditemukan. Isi FOLDER_REPO dengan lokasi folder "
                            "'hepatitis-leakage-reanalysis' (yang berisi folder data dan results).")
print("Folder kerja:", os.getcwd(), "| pelaksana:", NAMA_PELAKSANA)

Folder kerja: F:\ARTIKEL RAMEAN 2 -HEPATIITIS\hepatitis-leakage-reanalysis | pelaksana: Ade


In [2]:
# Instal sekali (hapus tanda # jika perlu), lalu restart kernel:
# !pip install pandas numpy scikit-learn imbalanced-learn xgboost scipy matplotlib

import os, sys, time, platform, hashlib, warnings, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from importlib.metadata import version, PackageNotFoundError

from sklearn.base import clone
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     GridSearchCV, ParameterGrid, cross_validate)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, precision_score, get_scorer
from joblib import Parallel, delayed
from imblearn.pipeline import Pipeline          # Pipeline yang mendukung SMOTE di dalam lipatan
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# ---------------- KONFIGURASI (ubah di sini saja) ----------------
FAST_MODE   = False      # True = uji coba cepat (ulangan sedikit, pohon sedikit). False = hasil final.
SEED        = 42         # seed utama; dicatat di laporan
N_SPLITS    = 5          # k pada stratified k-fold
N_REPEATS   = 2 if FAST_MODE else 10
INNER_FOLDS = 3          # validasi dalam untuk tuning
K_FEATURES  = 8          # jumlah fitur pada SelectKBest
N_TREES     = 50 if FAST_MODE else 200
N_BOOT      = 500 if FAST_MODE else 5000
DATA_DIR    = "data"
OUT_DIR     = "results"
MODELS      = ["NB", "DT", "KNN", "RF", "GB", "XGB"]
SENS_MODELS = ["NB", "DT", "GB"] if FAST_MODE else MODELS
os.makedirs(OUT_DIR, exist_ok=True)

SCORING = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",            # = sensitivitas pada kelas positif
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",   # PR-AUC
}
METRICS = list(SCORING)
print("Python", platform.python_version(), "| FAST_MODE =", FAST_MODE,
      "| N_REPEATS =", N_REPEATS, "| tanggal:", datetime.date.today())

Python 3.11.8 | FAST_MODE = False | N_REPEATS = 10 | tanggal: 2026-11-05


In [3]:
print("Folder kerja notebook:", os.getcwd())
TANGGAL_AKSES = "2026-10-08"   # tanggal unduh dari UCI

# Path relatif terhadap folder utama repositori
FILES = {"HCV": os.path.join("data", "hcv", "hcvdat0.csv"),
         "Hepatitis": os.path.join("data", "hepatitis", "hepatitis.data")}
for v in FILES.values():
    assert os.path.exists(v), f"File tidak ditemukan: {v}. Jalankan notebook dari folder utama repositori."

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(8192), b""):
            h.update(blok)
    return h.hexdigest()

cek = pd.DataFrame([{"dataset": k, "file": v, "ukuran_byte": os.path.getsize(v),
                     "sha256": sha256(v), "tanggal_akses": TANGGAL_AKSES}
                    for k, v in FILES.items()])
cek.to_csv(os.path.join(OUT_DIR, "checksum_data.csv"), index=False)
cek

# Data harus identik dengan eksperimen utama, karena hasil addendum dipasangkan dengan hasil lama.
SHA_ASLI = {"HCV": "1b62c8453b7098b695c549b19cfe070458ce6b909f0e990b6cfce7ab3c44c273",
            "Hepatitis": "9971f9d18abb4fc9dd6749eefb0f864b36fa7bdd59f54188c0b148a5c28214bc"}
for _, r in cek.iterrows():
    if r.sha256 != SHA_ASLI[r.dataset]:
        raise ValueError(
            f"File {r.dataset} ({os.path.abspath(r.file)}) BERBEDA dari data eksperimen utama.\n"
            "Kemungkinan file pernah dibuka lalu disimpan ulang dengan Excel (pemisah berubah menjadi ';'), "
            "atau notebook dijalankan dari folder lain.\n"
            "Solusi: jalankan notebook dari folder 'hepatitis-leakage-reanalysis' dan pakai file data asli dari repositori GitHub/Zenodo.")
print("Checksum data sesuai dengan eksperimen utama.")


Folder kerja notebook: F:\ARTIKEL RAMEAN 2 -HEPATIITIS\hepatitis-leakage-reanalysis
Checksum data sesuai dengan eksperimen utama.


In [4]:
HEP_COLS = ["Class", "AGE", "SEX", "STEROID", "ANTIVIRALS", "FATIGUE", "MALAISE", "ANOREXIA",
            "LIVER_BIG", "LIVER_FIRM", "SPLEEN_PALPABLE", "SPIDERS", "ASCITES", "VARICES",
            "BILIRUBIN", "ALK_PHOSPHATE", "SGOT", "ALBUMIN", "PROTIME", "HISTOLOGY"]

raw_hcv = pd.read_csv(FILES["HCV"])
raw_hcv = raw_hcv.rename(columns={raw_hcv.columns[0]: "X"})          # kolom ID tanpa nama
raw_hep = pd.read_csv(FILES["Hepatitis"],
                      header=None, names=HEP_COLS, na_values="?")

def periksa(df, nama, kolom_target):
    print("=" * 70)
    print(nama, "| dimensi:", df.shape)
    print("Baris duplikat:", int(df.duplicated().sum()))
    miss = df.isna().sum()
    print("Missing value per kolom:", miss[miss > 0].to_dict() or "tidak ada")
    print("Baris dengan >=1 missing:", int(df.isna().any(axis=1).sum()))
    print("Distribusi target:", df[kolom_target].value_counts().to_dict())
    print(df.dtypes.value_counts().to_dict())

periksa(raw_hcv, "UCI HCV", "Category")
periksa(raw_hep, "UCI Hepatitis", "Class")

UCI HCV | dimensi: (615, 14)
Baris duplikat: 0
Missing value per kolom: {'ALB': 1, 'ALP': 18, 'ALT': 1, 'CHOL': 10, 'PROT': 1}
Baris dengan >=1 missing: 26
Distribusi target: {'0=Blood Donor': 533, '3=Cirrhosis': 30, '1=Hepatitis': 24, '2=Fibrosis': 21, '0s=suspect Blood Donor': 7}
{dtype('float64'): 10, dtype('int64'): 2, dtype('O'): 2}
UCI Hepatitis | dimensi: (155, 20)
Baris duplikat: 0
Missing value per kolom: {'STEROID': 1, 'FATIGUE': 1, 'MALAISE': 1, 'ANOREXIA': 1, 'LIVER_BIG': 10, 'LIVER_FIRM': 11, 'SPLEEN_PALPABLE': 5, 'SPIDERS': 5, 'ASCITES': 5, 'VARICES': 5, 'BILIRUBIN': 6, 'ALK_PHOSPHATE': 29, 'SGOT': 4, 'ALBUMIN': 16, 'PROTIME': 67}
Baris dengan >=1 missing: 75
Distribusi target: {2: 123, 1: 32}
{dtype('float64'): 15, dtype('int64'): 5}


In [5]:
def siapkan_hcv(df):
    y = (~df["Category"].str.startswith("0")).astype(int)          # 0=Blood Donor & 0s=suspect -> 0
    X = df.drop(columns=["X", "Category"]).copy()
    X["Sex"] = (X["Sex"] == "m").astype(int)
    return X.reset_index(drop=True), y.reset_index(drop=True)

def siapkan_hep(df):
    y = (df["Class"] == 1).astype(int)                             # 1 = DIE = positif
    X = df.drop(columns=["Class"]).copy()
    return X.reset_index(drop=True), y.reset_index(drop=True)

DATASETS = {"HCV": siapkan_hcv(raw_hcv), "Hepatitis": siapkan_hep(raw_hep)}

baris = []
for nama, (X, y) in DATASETS.items():
    maj = max(y.mean(), 1 - y.mean())
    baris.append({"dataset": nama, "n": len(y), "fitur": X.shape[1], "positif": int(y.sum()),
                  "negatif": int((1 - y).sum()), "persen_positif": round(100 * y.mean(), 1),
                  "baseline_akurasi_mayoritas": round(100 * maj, 1)})
pd.DataFrame(baris)

,dataset,n,fitur,positif,negatif,persen_positif,baseline_akurasi_mayoritas
0,HCV,615,12,75,540,12.2,87.8
1,Hepatitis,155,19,32,123,20.6,79.4


In [6]:
def buat_cv(seed=SEED):
    return RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=seed)

print("Jumlah pembagian per model:", N_SPLITS * N_REPEATS)

Jumlah pembagian per model: 50


In [7]:
class XGBSeimbang(XGBClassifier):
    # class weight untuk XGBoost: rasio negatif/positif dihitung dari data latih lipatan saja
    def fit(self, X, y, **kw):
        y_arr = np.asarray(y)
        self.set_params(scale_pos_weight=(y_arr == 0).sum() / max((y_arr == 1).sum(), 1))
        return super().fit(X, y, **kw)

def klasifier(nama, balance, seed):
    cw = "balanced" if balance == "class_weight" else None
    if nama == "NB":
        return GaussianNB()
    if nama == "DT":
        return DecisionTreeClassifier(class_weight=cw, random_state=seed)
    if nama == "KNN":
        return KNeighborsClassifier()
    if nama == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, class_weight=cw, random_state=seed, n_jobs=1)
    if nama == "GB":
        return GradientBoostingClassifier(n_estimators=N_TREES, random_state=seed)
    if nama == "XGB":
        kelas = XGBSeimbang if balance == "class_weight" else XGBClassifier
        return kelas(n_estimators=N_TREES, eval_metric="logloss", random_state=seed, n_jobs=1, verbosity=0)
    raise ValueError(nama)

GRIDS = {
    "NB":  {"clf__var_smoothing": [1e-9, 1e-7, 1e-5]},
    "DT":  {"clf__max_depth": [3, 5, None], "clf__min_samples_leaf": [1, 5]},
    "KNN": {"clf__n_neighbors": [3, 5, 9]},
    "RF":  {"clf__max_depth": [None, 5], "clf__min_samples_leaf": [1, 3]},
    "GB":  {"clf__max_depth": [2, 3], "clf__learning_rate": [0.05, 0.2]},
    "XGB": {"clf__max_depth": [2, 4], "clf__learning_rate": [0.05, 0.2]},
}
BERLAKU = {"none": set(MODELS), "class_weight": {"DT", "RF", "XGB"}, "smote": set(MODELS)}

def buat_estimator(nama, balance, imputer="median", use_fs=True, n_fitur=None, tune=True, seed=SEED):
    # imputer: "median", "knn", atau "none" (data sudah tanpa NaN)
    langkah = []
    if imputer == "median":
        langkah.append(("imp", SimpleImputer(strategy="median")))
    elif imputer == "knn":
        langkah.append(("imp", KNNImputer(n_neighbors=5)))
    if nama == "KNN":
        langkah.append(("scale", StandardScaler()))
    if use_fs:
        langkah.append(("fs", SelectKBest(f_classif, k=min(K_FEATURES, n_fitur))))
    if balance == "smote":
        langkah.append(("smote", SMOTE(random_state=seed, k_neighbors=3)))
    langkah.append(("clf", klasifier(nama, balance, seed)))
    pipa = Pipeline(langkah)
    if tune:
        return GridSearchCV(pipa, GRIDS[nama], scoring="average_precision",
                            cv=StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed), n_jobs=1,
                            error_score="raise")
    return pipa

def rapikan(res, dataset, model, prosedur, config="utama"):
    d = pd.DataFrame({m: res[f"test_{m}"] for m in METRICS})
    d.insert(0, "split", np.arange(len(d)))
    d.insert(0, "config", config)
    d.insert(0, "prosedur", prosedur)
    d.insert(0, "model", model)
    d.insert(0, "dataset", dataset)
    return d

NAMA_PIPA = {"none": "P-A", "class_weight": "P-B", "smote": "P-C"}

def jalankan_pipa(X, y, dataset, nama, balance, imputer="median", seed=SEED, config="utama"):
    est = buat_estimator(nama, balance, imputer, use_fs=True, n_fitur=X.shape[1], seed=seed)
    res = cross_validate(est, X, y, cv=buat_cv(seed), scoring=SCORING, n_jobs=-1, error_score="raise")
    return rapikan(res, dataset, nama, NAMA_PIPA[balance], config)

In [8]:
def jalankan_semua_pipa(models=MODELS, imputer="median", seed=SEED, config="utama", balances=("none", "class_weight", "smote")):
    hasil = []
    for ds, (X, y) in DATASETS.items():
        for nama in models:
            for b in balances:
                if nama not in BERLAKU[b]:
                    continue
                t0 = time.time()
                hasil.append(jalankan_pipa(X, y, ds, nama, b, imputer, seed, config))
                print(f"[{ds}] {nama:3s} {NAMA_PIPA[b]}  ({time.time() - t0:.0f} dtk)")
    return pd.concat(hasil, ignore_index=True)

In [9]:
def impute_global(X, imputer="median"):
    imp = SimpleImputer(strategy="median") if imputer == "median" else KNNImputer(n_neighbors=5)
    return pd.DataFrame(imp.fit_transform(X), columns=X.columns)       # dipelajari dari seluruh baris (sebelum pembagian)

def smote_global(Xi, y, seed):
    Xr, yr = SMOTE(random_state=seed, k_neighbors=3).fit_resample(Xi, y)   # dilakukan pada seluruh data sebelum pembagian
    Xr, yr = np.asarray(Xr), np.asarray(yr)
    n = len(y)
    # imblearn menaruh baris asli di urutan awal, titik sintetis di belakang. Dicek agar penandaan baris asli pasti benar.
    assert np.allclose(Xr[:n], np.asarray(Xi, dtype=float)) and (yr[:n] == np.asarray(y)).all(), \
        "Urutan keluaran SMOTE tidak sesuai: baris asli tidak berada di awal."
    assert (yr[n:] == 1).all(), "Titik sintetis seharusnya semuanya kelas positif."
    return Xr, yr

SCORERS = {m: (get_scorer(s) if isinstance(s, str) else s) for m, s in SCORING.items()}

def _skor(est, X, y):
    return {f"test_{m}": SCORERS[m](est, X, y) for m in METRICS}

def _satu_lipatan_b(est, X_latih, y_latih, X_uji, y_uji):
    e = clone(est).fit(X_latih, y_latih)
    return _skor(e, X_uji, y_uji)

def cv_tambah_sintetis(est, Xi, y, X_sintetis, cv, kolom=None):
    """Pembagian dibuat dari baris asli (sama dengan P-C). Semua titik sintetis ditambahkan ke data latih;
    data uji hanya berisi baris asli. kolom: indeks fitur terpilih (untuk V4b)."""
    Xi = np.asarray(Xi, dtype=float); y = np.asarray(y)
    if kolom is not None:
        Xi, X_sintetis = Xi[:, kolom], X_sintetis[:, kolom]
    y_sint = np.ones(len(X_sintetis), dtype=int)
    tugas = (delayed(_satu_lipatan_b)(est, np.vstack([Xi[tr], X_sintetis]), np.concatenate([y[tr], y_sint]),
                                      Xi[te], y[te]) for tr, te in cv.split(Xi, y))
    hasil = Parallel(n_jobs=-1)(tugas)
    return {k: np.array([h[k] for h in hasil]) for k in hasil[0]}

def jalankan_variasi(X, y, dataset, nama, skenario, imputer="median", seed=SEED, config="utama"):
    cv = buat_cv(seed)
    n = len(y)
    if skenario == "V1":
        Xi = impute_global(X, imputer)
        est = buat_estimator(nama, "smote", "none", use_fs=True, n_fitur=Xi.shape[1], seed=seed)
        res = cross_validate(est, Xi, y, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    elif skenario == "V2":
        Xr, yr = smote_global(impute_global(X, imputer), y, seed)
        est = buat_estimator(nama, "none", "none", use_fs=True, n_fitur=Xr.shape[1], seed=seed)
        res = cross_validate(est, Xr, yr, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    elif skenario == "V2b":
        Xi = impute_global(X, imputer)
        Xr, yr = smote_global(Xi, y, seed)
        est = buat_estimator(nama, "none", "none", use_fs=True, n_fitur=Xi.shape[1], seed=seed)
        res = cv_tambah_sintetis(est, Xi, y, Xr[n:], cv)
    elif skenario == "V3":
        Xi = impute_global(X, imputer)
        terpilih = SelectKBest(f_classif, k=min(K_FEATURES, Xi.shape[1])).fit(Xi, y).get_support()   # fitur dipilih dari seluruh data
        Xs = X.loc[:, terpilih]
        est = buat_estimator(nama, "smote", imputer, use_fs=False, seed=seed)
        res = cross_validate(est, Xs, y, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
    elif skenario in ("V4", "V4b"):
        Xi = impute_global(X, imputer)
        Xr, yr = smote_global(Xi, y, seed)
        terpilih = SelectKBest(f_classif, k=min(K_FEATURES, Xr.shape[1])).fit(Xr, yr).get_support()  # fitur dipilih dari seluruh data
        kolom = np.flatnonzero(terpilih)
        terbaik = None
        for params in ParameterGrid(GRIDS[nama]):               # hiperparameter dipilih berdasarkan skor data uji
            est = buat_estimator(nama, "none", "none", use_fs=False, tune=False, seed=seed).set_params(**params)
            if skenario == "V4":
                r = cross_validate(est, Xr[:, kolom], yr, cv=cv, scoring=SCORING, n_jobs=-1, error_score="raise")
            else:
                r = cv_tambah_sintetis(est, Xi, y, Xr[n:], cv, kolom=kolom)
            skor = r["test_average_precision"].mean()
            if terbaik is None or skor > terbaik[0]:
                terbaik = (skor, r)
        res = terbaik[1]
    else:
        raise ValueError(skenario)
    return rapikan(res, dataset, nama, skenario, config)

SEMUA_VARIASI = ("V1", "V2", "V2b", "V3", "V4", "V4b")

def jalankan_semua_variasi(models=MODELS, skenario=SEMUA_VARIASI, imputer="median", seed=SEED, config="utama"):
    hasil = []
    for ds, (X, y) in DATASETS.items():
        for nama in models:
            for s in skenario:
                t0 = time.time()
                hasil.append(jalankan_variasi(X, y, ds, nama, s, imputer, seed, config))
                print(f"[{ds}] {nama:3s} {s:3s}  ({time.time() - t0:.0f} dtk)")
    return pd.concat(hasil, ignore_index=True)

In [10]:
def ttest_terkoreksi(d, n_splits=N_SPLITS):
    """Corrected resampled t-test (Nadeau & Bengio 2003) untuk selisih per lipatan dari repeated k-fold CV."""
    d = np.asarray(d, dtype=float)
    J = len(d)
    rasio = 1.0 / (n_splits - 1)                     # n_uji / n_latih
    rata, var = d.mean(), d.var(ddof=1)
    se = np.sqrt((1.0 / J + rasio) * var)
    tkrit = stats.t.ppf(0.975, J - 1)
    if se == 0:                                      # semua selisih identik
        return rata, rata, rata, np.nan, np.nan
    t = rata / se
    p = 2 * stats.t.sf(abs(t), J - 1)
    return rata, rata - tkrit * se, rata + tkrit * se, t, p

def holm(p):
    p = np.asarray(p, dtype=float); out = np.full_like(p, np.nan)
    ok = ~np.isnan(p); q = p[ok]; m = len(q)
    if m == 0: return out
    idx = np.argsort(q)
    adj = np.maximum.accumulate((m - np.arange(m)) * q[idx])
    res = np.empty(m); res[idx] = np.minimum(adj, 1.0)
    out[ok] = res
    return out

def bh(p):
    p = np.asarray(p, dtype=float); out = np.full_like(p, np.nan)
    ok = ~np.isnan(p); q = p[ok]; m = len(q)
    if m == 0: return out
    idx = np.argsort(q)
    adj = q[idx] * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    res = np.empty(m); res[idx] = np.minimum(adj, 1.0)
    out[ok] = res
    return out

BERPASANGAN = ("V1", "V2b", "V3", "V4b")

def analisis_selisih(df, ref="P-C", metrik=("accuracy", "balanced_accuracy", "recall", "f1", "average_precision", "roc_auc"),
                     skenario=SEMUA_VARIASI, alpha=0.05):
    baris = []
    for (ds, model, cfg), g in df.groupby(["dataset", "model", "config"]):
        c = g[g.prosedur == ref].sort_values("split")
        if c.empty:
            continue
        for k in skenario:
            w = g[g.prosedur == k].sort_values("split")
            if w.empty:
                continue
            for m in metrik:
                a, b = c[m].to_numpy(), w[m].to_numpy()
                if k in BERPASANGAN:
                    assert len(a) == len(b), "Jumlah lipatan berbeda; pembagian tidak sejajar."
                    est, lo, hi, t, p = ttest_terkoreksi(b - a)
                    uji = "corrected resampled t-test (berpasangan)"
                else:
                    est, lo, hi, t, p = b.mean() - a.mean(), np.nan, np.nan, np.nan, np.nan
                    uji = "deskriptif (tidak berpasangan)"
                baris.append({"dataset": ds, "model": model, "config": cfg, "skenario": k, "metrik": m,
                              "selisih_rata2": est, "ci95_low": lo, "ci95_high": hi, "t": t,
                              "p_value": p, "uji": uji})
    out = pd.DataFrame(baris)
    # koreksi uji ganda: per konfigurasi dan per metrik
    out["p_holm"] = np.nan; out["p_bh"] = np.nan
    for _, idx in out.groupby(["config", "metrik"]).groups.items():
        out.loc[idx, "p_holm"] = holm(out.loc[idx, "p_value"])
        out.loc[idx, "p_bh"] = bh(out.loc[idx, "p_value"])
    def label(r):
        if np.isnan(r.p_value):
            return "deskriptif" if r.uji.startswith("deskriptif") else "tidak berbeda"
        if r.p_holm < alpha:
            return "lebih tinggi" if r.selisih_rata2 > 0 else "lebih rendah"
        return "tidak berbeda"
    out["kesimpulan_holm"] = out.apply(label, axis=1)
    return out

In [11]:
UTAMA = pd.read_csv(os.path.join(OUT_DIR, "hasil_semua_mentah.csv"))   # hasil eksperimen utama
print(UTAMA.shape, sorted(UTAMA.prosedur.unique()))

(5100, 12) ['P-A', 'P-B', 'P-C', 'V1', 'V2', 'V2b', 'V3', 'V4', 'V4b']


In [12]:
# Hasil Bagian C dan D dari run sebelumnya
hasil_k = pd.read_csv(os.path.join(OUT_DIR, "revisi_C_k_mentah.csv"))
hasil_d = pd.read_csv(os.path.join(OUT_DIR, "revisi_D_v2b80_mentah.csv"))
print("hasil_k:", hasil_k.shape, sorted(hasil_k.config.unique()), "| hasil_d:", hasil_d.shape)

hasil_k: (3600, 12) ['k=3', 'k=5', 'k=8'] | hasil_d: (600, 12)


## Bagian E. Pembanding dari lingkungan yang sama (versi paket)
Eksperimen utama dijalankan dengan scikit-learn 1.3.0. Jika addendum dijalankan dengan versi lain, angka k = 8 di Bagian C dan V2b (100%) di Bagian D berasal dari versi yang berbeda dengan angka baru. Bagian ini menjalankan ulang **P-C, V2b, dan V3 (k = 8)** dengan lingkungan sekarang, lalu:
1. membandingkannya dengan hasil utama (cek reprodusibilitas lintas versi), dan
2. menghitung ulang tabel C dan D memakai pembanding dari lingkungan yang sama.

Bagian ini otomatis dilewati jika versi scikit-learn sama dengan eksperimen utama. Perkiraan waktu: ±25 menit.

In [13]:
import json, sklearn
VERSI_UTAMA = json.load(open(os.path.join(OUT_DIR, "konfigurasi_eksperimen.json")))["versi_paket"]["scikit-learn"]
print("scikit-learn eksperimen utama:", VERSI_UTAMA, "| sekarang:", sklearn.__version__)

if sklearn.__version__ == VERSI_UTAMA:
    print("Versi sama: pembanding dari hasil utama sudah sah, Bagian E dilewati.")
else:
    K_FEATURES = 8
    ulang = [jalankan_semua_pipa(balances=("smote",), config="ulang"),
             jalankan_semua_variasi(skenario=("V2b", "V3"), config="ulang")]
    ulang = pd.concat(ulang, ignore_index=True)
    ulang.to_csv(os.path.join(OUT_DIR, "revisi_E_ulang_k8_mentah.csv"), index=False)

    # E1. Reprodusibilitas: hasil ulang vs hasil utama (lipatan sama)
    m4 = ["f1", "average_precision", "balanced_accuracy", "roc_auc"]
    a = UTAMA[UTAMA.prosedur.isin(["P-C", "V2b", "V3"])].set_index(["dataset", "model", "prosedur", "split"])[m4]
    b = ulang.set_index(["dataset", "model", "prosedur", "split"])[m4]
    beda = (b - a).dropna()
    repro = beda.abs().groupby(level=["dataset", "prosedur"]).max().round(4)
    repro.to_csv(os.path.join(OUT_DIR, "revisi_E1_reprodusibilitas.csv"))
    print("Selisih maksimum per lipatan (ulang - utama):"); display(repro)

    # E2. Tabel C dengan k = 8 dari lingkungan yang sama
    hasil_k2 = pd.concat([hasil_k[hasil_k.config != "k=8"],
                          ulang[ulang.prosedur.isin(["P-C", "V3"])].assign(config="k=8")], ignore_index=True)
    sel_k2 = analisis_selisih(hasil_k2, skenario=("V3",))
    sel_k2.to_csv(os.path.join(OUT_DIR, "revisi_E2_selisih_V3_per_k_samaversi.csv"), index=False)
    print("V3 - P-C per k (semua dari lingkungan yang sama):")
    display(sel_k2[sel_k2.metrik.isin(["f1", "average_precision", "balanced_accuracy"])]
            .groupby(["dataset", "metrik", "config"])["selisih_rata2"].mean().unstack("config").round(3))
    print("Jumlah model F1 V3 lebih tinggi nyata (Holm):")
    display(sel_k2[sel_k2.metrik == "f1"].assign(n=lambda d: d.kesimpulan_holm == "lebih tinggi")
            .groupby(["dataset", "config"])["n"].sum().unstack("config"))

    # E3. V2b-80 vs V2b-100 dari lingkungan yang sama
    a2 = ulang[ulang.prosedur == "V2b"].set_index(["dataset", "model", "split"])[m4]
    b2 = hasil_d.set_index(["dataset", "model", "split"])[m4]
    pc2 = ulang[ulang.prosedur == "P-C"].set_index(["dataset", "model", "split"])[m4]
    v2 = UTAMA[UTAMA.prosedur == "V2"].groupby(["dataset", "model"])[m4].mean()     # V2 tidak berpasangan (deskriptif)
    pc_u = UTAMA[UTAMA.prosedur == "P-C"].groupby(["dataset", "model"])[m4].mean()
    t = pd.DataFrame({
        "V2b100_minus_PC": (a2 - pc2).groupby(level=["dataset", "model"]).mean().stack(),
        "V2b80_minus_PC": (b2 - pc2).groupby(level=["dataset", "model"]).mean().stack(),
    })
    t["V2b80_minus_V2b100"] = t.V2b80_minus_PC - t.V2b100_minus_PC
    t.index.names = ["dataset", "model", "metrik"]
    ring = t.groupby(level=["dataset", "metrik"]).mean()
    total_v2 = (v2 - pc_u).groupby(level="dataset").mean().stack(); total_v2.index.names = ["dataset", "metrik"]
    ring["total_V2_minus_PC (utama)"] = total_v2
    ring["porsi_V2b100_%"] = 100 * ring.V2b100_minus_PC / ring["total_V2_minus_PC (utama)"]
    ring["porsi_V2b80_%"] = 100 * ring.V2b80_minus_PC / ring["total_V2_minus_PC (utama)"]
    ring.round(3).to_csv(os.path.join(OUT_DIR, "revisi_E3_dekomposisi_samaversi.csv"))
    t.round(4).to_csv(os.path.join(OUT_DIR, "revisi_E3_per_model_samaversi.csv"))
    print("V2b 100% vs 80% (lingkungan sama):"); display(ring.round(3))

scikit-learn eksperimen utama: 1.3.0 | sekarang: 1.9.1
[HCV] NB  P-C  (13 dtk)
[HCV] DT  P-C  (8 dtk)
[HCV] KNN P-C  (5 dtk)
[HCV] RF  P-C  (111 dtk)
[HCV] GB  P-C  (85 dtk)
[HCV] XGB P-C  (21 dtk)
[Hepatitis] NB  P-C  (4 dtk)
[Hepatitis] DT  P-C  (7 dtk)
[Hepatitis] KNN P-C  (4 dtk)
[Hepatitis] RF  P-C  (97 dtk)
[Hepatitis] GB  P-C  (54 dtk)
[Hepatitis] XGB P-C  (15 dtk)
[HCV] NB  V2b  (2 dtk)
[HCV] NB  V3   (4 dtk)
[HCV] DT  V2b  (4 dtk)
[HCV] DT  V3   (7 dtk)
[HCV] KNN V2b  (4 dtk)
[HCV] KNN V3   (5 dtk)
[HCV] RF  V2b  (124 dtk)
[HCV] RF  V3   (109 dtk)
[HCV] GB  V2b  (99 dtk)
[HCV] GB  V3   (89 dtk)
[HCV] XGB V2b  (21 dtk)
[HCV] XGB V3   (18 dtk)
[Hepatitis] NB  V2b  (4 dtk)
[Hepatitis] NB  V3   (3 dtk)
[Hepatitis] DT  V2b  (3 dtk)
[Hepatitis] DT  V3   (6 dtk)
[Hepatitis] KNN V2b  (2 dtk)
[Hepatitis] KNN V3   (4 dtk)
[Hepatitis] RF  V2b  (96 dtk)
[Hepatitis] RF  V3   (97 dtk)
[Hepatitis] GB  V2b  (53 dtk)
[Hepatitis] GB  V3   (52 dtk)
[Hepatitis] XGB V2b  (10 dtk)
[Hepatitis] XGB V

f1  average_precision  balanced_accuracy  roc_auc
dataset   prosedur                                                    
HCV       P-C       0.0                0.0                0.0      0.0
          V2b       0.0                0.0                0.0      0.0
          V3        0.0                0.0                0.0      0.0
Hepatitis P-C       0.0                0.0                0.0      0.0
          V2b       0.0                0.0                0.0      0.0
          V3        0.0                0.0                0.0      0.0

V3 - P-C per k (semua dari lingkungan yang sama):


config                         k=3    k=5    k=8
dataset   metrik                                
HCV       average_precision  0.000  0.000  0.021
          balanced_accuracy  0.000 -0.001  0.018
          f1                 0.000  0.001  0.028
Hepatitis average_precision  0.030  0.005  0.012
          balanced_accuracy  0.008  0.002  0.007
          f1                 0.014  0.002  0.010

Jumlah model F1 V3 lebih tinggi nyata (Holm):


config,k=3,k=5,k=8
dataset,,,
HCV,0,0,0
Hepatitis,0,0,0


V2b 100% vs 80% (lingkungan sama):


V2b100_minus_PC  V2b80_minus_PC  V2b80_minus_V2b100  total_V2_minus_PC (utama)  porsi_V2b100_%  porsi_V2b80_%
dataset   metrik                                                                                                                          
HCV       average_precision            0.073           0.078               0.006                      0.165          44.070         47.433
          balanced_accuracy            0.058           0.058              -0.000                      0.061          95.287         94.986
          f1                           0.068           0.077               0.009                      0.183          36.953         42.040
          roc_auc                      0.033           0.034               0.001                      0.037          89.395         91.491
Hepatitis average_precision            0.060           0.045              -0.015                      0.303          19.755         14.928
          balanced_accuracy            0.044           0.037              -0.007                      0.115          38.286         32.211
          f1                           0.056           0.049              -0.007                      0.298          18.716         16.475
          roc_auc                      0.048           0.038              -0.009                      0.101          47.138         37.836

In [14]:
# Simpan versi paket untuk addendum
import json
from importlib.metadata import version, PackageNotFoundError
v = {}
for p in ["numpy", "pandas", "scipy", "scikit-learn", "imbalanced-learn", "xgboost"]:
    try: v[p] = version(p)
    except PackageNotFoundError: v[p] = "tidak ditemukan"
info = {"pelaksana": NAMA_PELAKSANA, "python": platform.python_version(), "tanggal_jalan": str(datetime.date.today()), "SEED": SEED,
        "N_REPEATS": N_REPEATS, "FAST_MODE": FAST_MODE, "versi_paket": v}
json.dump(info, open(os.path.join(OUT_DIR, "revisi_E_konfigurasi.json"), "w"), indent=2)
info

{'pelaksana': 'Ade',
 'python': '3.11.8',
 'tanggal_jalan': '2026-11-05',
 'SEED': 42,
 'N_REPEATS': 10,
 'FAST_MODE': False,
 'versi_paket': {'numpy': '1.26.4',
  'pandas': '2.2.1',
  'scipy': '1.12.0',
  'scikit-learn': '1.9.1',
  'imbalanced-learn': '0.14.2',
  'xgboost': '2.1.4'}}